# Parameterised pentagon spiral winding

A planar spiral winding whose turns follow a rounded polygon rather than a circle.

The base shape is a **pentagon given directly as five vertices on the x-y plane**, and
those vertices are the centre poles for the cornering arcs. The wire walks outwards turn
by turn: an arc around a pole, a straight run along the offset edge to the next pole, and
so on, so that a full lap of five corners advances the winding by exactly one `pitch`.

Nothing below assumes the pentagon is regular. Any convex polygon in vertex order will do,
so the shape can be made irregular just by editing the coordinates.

This alternates the `a` and `l` directives:

- `a x_z y_z 1 1` rounds a corner about the pole at `(x_z, y_z)` in the positive direction,
  linearly interpolating the radius from that pole as it sweeps, which is what gives the
  turn-to-turn advance.
- `l 1` runs straight along the edge, offset outwards by the current radius.

The `a` directive is what makes this possible: `s` can only spiral about the origin, so a
five-pole winding would otherwise need the whole pattern translated between corners.


## 1. Define the winding

### Parameters

The five vertices are listed directly, in order round the shape. They default to a regular
pentagon of 30 mm circumradius, but any convex polygon works -- edit the coordinates to
make the shape irregular. Winding direction is handled automatically, so the vertices may
be listed either clockwise or counter-clockwise.

In [ ]:
import numpy as np

# ---- base shape: the five vertices on the x-y plane, in order round the shape ----
# These are the centre poles for the cornering arcs. Coordinates in metres.
vertices = np.array([
    [ 0.030000,  0.000000],
    [ 0.009271,  0.028532],
    [-0.024271,  0.017634],
    [-0.024271, -0.017634],
    [ 0.009271, -0.028532],
])

# ---- winding parameters ----
r_cs       = 0.00045  # wire cross-section radius (m)
pitch      = 0.0015   # radial advance per turn (m)
n_turns    = 6        # number of turns
r_corner_0 = 0.0015   # corner arc radius of the innermost turn (m)

# ---- output ----
file_def = "wire_def"   # wire definition file written for the solver
file_out = "pentagon"   # stem of the solver's output files

### Build the trajectory

Each corner arc is centred on a vertex and sweeps the exterior angle there, growing the
radius as it goes; each straight run is the edge offset outwards by the radius the
preceding arc finished at. The radius increment is shared between the five corners in
proportion to their exterior angles, so the radius grows uniformly with turned angle
whatever the shape. Since the exterior angles of any convex polygon sum to `2*pi`, one lap
adds exactly one `pitch`. For a regular pentagon this is just `pitch / 5` per corner.

In [ ]:
n_sides = len(vertices)

# The construction below walks counter-clockwise, taking outward normals and positive
# arcs. If the vertices were listed clockwise, reverse them so that both hold.
signed_area = 0.5 * np.sum(vertices[:, 0] * np.roll(vertices[:, 1], -1)
                           - np.roll(vertices[:, 0], -1) * vertices[:, 1])
if signed_area < 0:
    vertices = vertices[::-1]
    print("vertices were listed clockwise, reversed to counter-clockwise")

if pitch <= 2 * r_cs:
    print("warning: pitch %g m is not greater than the wire diameter %g m, turns will overlap"
          % (pitch, 2 * r_cs))

# Outward unit normal of each edge i, running from vertex i to vertex i + 1.
edges = np.roll(vertices, -1, axis=0) - vertices
side_lengths = np.linalg.norm(edges, axis=1)
normals = np.stack([edges[:, 1], -edges[:, 0]], axis=1) / side_lengths[:, None]

# Exterior angle turned at vertex i, i.e. from the normal of the incoming edge to that of
# the outgoing edge. This is the angle the arc about vertex i has to sweep.
normal_angles = np.arctan2(normals[:, 1], normals[:, 0])
exterior_angles = (normal_angles - np.roll(normal_angles, 1)) % (2 * np.pi)
if np.any(exterior_angles > np.pi):
    print("warning: the shape is not convex, the offset construction below will self-intersect")

# Share the per-turn advance between the corners in proportion to the angle each turns.
dr = pitch * exterior_angles / (2 * np.pi)

# ---- walk the spiral: arc round a pole, straight along the offset edge, repeat ----
# The point arriving at vertex i sits at radius r along normals[i - 1]; the arc about
# vertex i sweeps it round to radius r + dr[i] along normals[i].
r = r_corner_0
length_analytic = 0.0
commands = [("l", None, vertices[0] + r * normals[-1])]
for turn in range(n_turns):
    for i in range(n_sides):
        r_next = r + dr[i]
        commands.append(("a", vertices[i], vertices[i] + r_next * normals[i]))
        commands.append(("l", None, vertices[(i + 1) % n_sides] + r_next * normals[i]))
        # Arc at the mean radius, then the straight run, which for an outward offset of a
        # convex polygon is exactly the length of the edge itself.
        length_analytic += 0.5 * (r + r_next) * exterior_angles[i] + side_lengths[i]
        r = r_next

print("%d turns, %d commands, corner radius %.4f m -> %.4f m"
      % (n_turns, len(commands), r_corner_0, r))

### Write the wire definition file

In [ ]:
with open(file_def, "w") as file_out_def:
    file_out_def.write("1\r\n")
    file_out_def.write("%g %d\r\n" % (r_cs, len(commands)))
    for mode, pole, point in commands:
        if mode == "a":
            file_out_def.write("a %.9g %.9g 1 1\r\n" % (pole[0], pole[1]))
        else:
            file_out_def.write("l 1\r\n")
        file_out_def.write("%.9g %.9g 0\r\n" % (point[0], point[1]))

print("wrote %s\n" % file_def)
print("".join(open(file_def).readlines()[:7]).replace("\r", ""), "...", sep="")

### Preview the key points

The solver writes the segmented trajectory to `pentagon_0.trj`, so the winding itself is
plotted after the solver has run in section 2. Here we just preview the key points that
were written to the definition file, with the base shape dashed.

In [ ]:
import matplotlib.pyplot as plt

key_points = np.array([point for _, _, point in commands])

figure, ax = plt.subplots(figsize=(6, 6))
ax.plot(key_points[:, 0], key_points[:, 1], "o-", ms=2, lw=0.8, label="key points")
ax.plot(np.append(vertices[:, 0], vertices[0, 0]), np.append(vertices[:, 1], vertices[0, 1]),
        "r--", lw=0.8, label="base shape (corner poles)")
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Key points of the pentagon winding")
ax.legend(loc="upper right", fontsize=8)
plt.show()

## 2. Solve for self-inductance and total length

The solver is run in its default `-l` mode, which writes `pentagon.lmat`. For a single
wire that file holds a 1x1 L matrix -- the self-inductance in henries -- followed by a
last line giving the total wire length in metres.

In [ ]:
import os

status = os.system("../../../build/ivsolver %s %s" % (file_def, file_out))
if status != 0:
    raise RuntimeError("ivsolver failed with status %d" % status)

# The .lmat file is n rows of the L matrix followed by one row of wire lengths, so for a
# single wire it is one row holding L and one holding the length. Parse it by row rather
# than with loadtxt, which would flatten the 1x1 case into a single vector.
rows = [line.split() for line in open(file_out + ".lmat") if line.strip()]
n_wires = len(rows) - 1
l_matrix = np.array([[float(value) for value in row] for row in rows[:n_wires]])
lengths = np.array([float(value) for value in rows[n_wires]])

self_inductance = l_matrix[0, 0]
total_length = lengths[0]

print("self-inductance: %.4f uH" % (self_inductance * 1e6))
print("total length:    %.4f m" % total_length)

### Cross-check the length

`length_analytic` was accumulated alongside the trajectory in section 1 from the geometry
itself, so it is an independent check on the solver. It is a first-order figure -- it
takes each arc at its mean radius rather than integrating the radial growth -- and the
solver in turn approximates the arcs by chords of one wire diameter, so the two agree to
within their respective discretisation errors rather than exactly.

In [ ]:
print("analytic length: %.6f m" % length_analytic)
print("solver length:   %.6f m" % total_length)
print("difference:      %.4f %%" % (100 * (total_length - length_analytic) / length_analytic))

### Plot the segmented trajectory

`pentagon_0.trj` is the trajectory the solver actually integrated over, segmented into
steps of one wire diameter.

In [ ]:
trajectory = np.loadtxt(file_out + "_0.trj")

figure, ax = plt.subplots(figsize=(6, 6))
ax.plot(trajectory[:, 0], trajectory[:, 1], lw=0.9)
ax.plot(np.append(vertices[:, 0], vertices[0, 0]), np.append(vertices[:, 1], vertices[0, 1]),
        "r--", lw=0.8, label="base shape (corner poles)")
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("%d-turn pentagon winding, %.2f uH, %.3f m"
             % (n_turns, self_inductance * 1e6, total_length))
ax.legend(loc="upper right", fontsize=8)
figure.savefig("winding.png", dpi=110, bbox_inches="tight")
plt.show()

print("%d trajectory points" % len(trajectory))